# Notebook de Pruebas Integrales

Este notebook prueba los módulos actualizados en `src/`:
- `elhd.py`
- `battery.py`
- `layout.py`
- `__init__.py` (clase `Mine`)
- `timeseries.py`

Contiene las siguientes secciones:
1. Ajuste de rutas e importaciones.
2. Carga de datos desde `elmo_data.xlsx` y creación del objeto `Mine`.
3. Pruebas de getters en `ELHD`, `Battery` y `Layout`.
4. Filtrado de LHDs por tecnología.
5. Carga de datos desde `time_series.xlsx` y creación de `Timeseries`.
6. Pruebas de mapeos de asignación de nodos y baterías.
7. Cálculo y verificación de `trips`.


In [1]:
# 1) Ajuste de rutas e importaciones tal como en el notebook original
from typing import NamedTuple
from IPython import get_ipython
import sys, os
from os.path import join
import pyomo.environ as pyo
# Habilitar autoreload
ipython = get_ipython()
ipython.magic('load_ext autoreload')
ipython.magic('autoreload 2')
sys.path.append(os.path.abspath(os.path.join('..')))


from src.io.reader import Reader, Series
from src import mine
from src.time_series  import  timeseries 
from src.optimization import functions 
import pandas as pd
import numpy as np  


C:\Users\dlira\AppData\Local\Temp\ipykernel_15352\138770338.py:9: DeprecationWarning: `magic(...)` is deprecated since IPython 0.13 (warning added in 8.1), use run_line_magic(magic_name, parameter_s).
  ipython.magic('load_ext autoreload')
C:\Users\dlira\AppData\Local\Temp\ipykernel_15352\138770338.py:10: DeprecationWarning: `magic(...)` is deprecated since IPython 0.13 (warning added in 8.1), use run_line_magic(magic_name, parameter_s).
  ipython.magic('autoreload 2')


In [2]:
class Parameters(NamedTuple):
    """ Define a MOC class which contains default values
    from program model simulation 
    
    """
    selected = 'Case1'
    model = join('..', 'data', r'C:\Users\dlira\OneDrive\Escritorio\ELMO\Modelos\ELMO-UG-Paper\data\data_electrico_var_3turnos_machali_verano\elmo_data.xlsx')
    series = join('..', 'data', r'C:\Users\dlira\OneDrive\Escritorio\ELMO\Modelos\ELMO-UG-Paper\data\data_electrico_var_3turnos_machali_verano\time_series.xlsx')

args = Parameters()
model = Reader(args.model, start_in=1)
series = Series(args.series)

time_series=timeseries.Timeseries(series, [1],0.5)
mine_system = mine.Mine(model)

time_series.get_trips(mine_system)
time_series.get_node_assignment(mine_system.get_system_lhds())
time_series.get_elhd_at_node(mine_system.get_system_nodes())

time_series.get_battery_assignment(mine_system.get_system_lhds())
time_series.get_elhd_with_battery(mine_system.get_system_batteries())

model_pyo = pyo.ConcreteModel()

from src.optimization.functions import OptSets, OptParameters, BoundRules

opt_sets   = OptSets(mine_system, time_series)
opt_params = OptParameters(mine_system, time_series)
bounds     = BoundRules(mine_system, time_series)


#opt_sets.build_sets(model_pyo)
#opt_params.build_parameters(model_pyo)
#bounds.build_all_variables(model_pyo)


In [3]:
for i in mine_system.get_system_lhds():
    litros = time_series.get_diesel_consumption("PEX_10", i)
    print(f"Consumo de diésel para LHD_3 en Nodo_A: {litros:.2f} litros")

Consumo de diésel para LHD_3 en Nodo_A: 0.00 litros
Consumo de diésel para LHD_3 en Nodo_A: 0.00 litros
Consumo de diésel para LHD_3 en Nodo_A: 0.00 litros
Consumo de diésel para LHD_3 en Nodo_A: 0.00 litros


In [4]:
print(time_series.get_trips(mine_system))

                travel_duration n_trips energy_consumption  diesel_consumption
elhd     node                                                                 
LH518B_1 PEX_1                1    13.0           1.204296                 0.0
         PEX_10               1    10.0           2.274647                 0.0
         PEX_11               1    12.0           1.632437                 0.0
         PEX_12               1    11.0           1.900024                 0.0
         PEX_13               1    10.0           2.167612                 0.0
...                         ...     ...                ...                 ...
LH518B_4 PEX_5                1    10.0           2.274647                 0.0
         PEX_6                1    13.0           1.204296                 0.0
         PEX_7                1    12.0           1.471884                 0.0
         PEX_8                1    12.0           1.739472                 0.0
         PEX_9                1    11.0            2

In [5]:
def get_trips_for_single_lhd(time_series, mine_system, elhd_name):
    """
    Calcula el número de viajes y consumo de energía para un único LHD en todos los nodos.
    """
    nodes = mine_system.get_system_nodes()  # Lista de todos los nodos
    index = pd.Index(nodes, name='node')
    trips = pd.DataFrame(index=index, columns=['travel_duration', 'n_trips', 'energy_consumption'])

    for node in nodes:
        distance_outbound = mine_system.layout.get_distance_to_d_node_outbound(node)
        distance_return = mine_system.layout.get_distance_to_d_node_return(node)
        tilt = mine_system.layout.get_tilt(node)

        # Usamos el método del ELHD para obtener duración y energía
        travel_dur_hours, energy_per_trip = mine_system.elhd.get_total_trips_info(
            distance_outbound, distance_return, tilt, elhd_name, time_series.delta_t
        )
        
        if travel_dur_hours <= time_series.delta_t:
            n_trips = np.floor(time_series.delta_t / travel_dur_hours)
        else:
            n_trips = 1

        trips.loc[node, 'n_trips'] = n_trips
        trips.loc[node, 'travel_duration'] = travel_dur_hours
        trips.loc[node, 'energy_consumption'] = energy_per_trip

    return trips

# Ejemplo de uso:
# Supón que el nombre del LHD es 'LHD-01'
lhd_name = 'LH518B_1'
single_lhd_trips = get_trips_for_single_lhd(time_series, mine_system, lhd_name)

# Ordenar el DataFrame numéricamente por el número en el nombre del nodo
single_lhd_trips_sorted = single_lhd_trips.sort_index(
    key=lambda x: x.str.extract(r'(\d+)').astype(int).iloc[:, 0]
)

# Mostrar todas las filas ordenadas
pd.set_option('display.max_rows', None)
print(single_lhd_trips_sorted)


       travel_duration n_trips energy_consumption
node                                             
PEX_1         0.036049    13.0           1.204296
PEX_2         0.038716    12.0           1.471884
PEX_3         0.041383    12.0           1.739472
PEX_4         0.044049    11.0            2.00706
PEX_5         0.046716    10.0           2.274647
PEX_6         0.036049    13.0           1.204296
PEX_7         0.038716    12.0           1.471884
PEX_8         0.041383    12.0           1.739472
PEX_9         0.044049    11.0            2.00706
PEX_10        0.046716    10.0           2.274647
PEX_11        0.040316    12.0           1.632437
PEX_12        0.042983    11.0           1.900024
PEX_13        0.045649    10.0           2.167612
PEX_14        0.048316    10.0             2.4352
PEX_15        0.050983     9.0           2.702788
PEX_16        0.040316    12.0           1.632437
PEX_17        0.042983    11.0           1.900024
PEX_18        0.045649    10.0           2.167612


In [22]:
# -------------------------------
# Bloque de prueba para 5 recorridos
# -------------------------------

# Selección de 5 nodos distintos (puedes cambiarlos por los que desees)
nodos_prueba = mine_system.get_system_nodes()[:5]  # Primeros 5 nodos del layout
vehiculo_prueba = mine_system.get_system_lhds()[0]  # Primer LHD disponible
print(f"Vehículo seleccionado: {vehiculo_prueba}")
# Intentar obtener delta_t desde timeseries o definirlo manualmente

delta_t = 0.5 # horas (cambiar si tu simulación usa otro)

resultados = []

for nodo in nodos_prueba:
    dist_ida = mine_system.layout.get_distance_to_d_node_outbound(nodo)
    dist_vuelta = mine_system.layout.get_distance_to_d_node_return(nodo)
    tilt = mine_system.layout.get_tilt(nodo)

    t_viaje, energia_total = mine_system.elhd.get_total_trips_info(
        dist_ida, dist_vuelta, tilt, vehiculo_prueba, delta_t
    )

    speed_m_s = mine_system.elhd.get_speed(vehiculo_prueba) * (1000/3600)
    acceleration = mine_system.elhd.get_acceleration(vehiculo_prueba)
    aux_power_kW = mine_system.elhd.get_aux_power(vehiculo_prueba)
    print(aux_power_kW)
    hydraulic_power_kW = mine_system.elhd.get_hydraulic_power(vehiculo_prueba)
    print(hydraulic_power_kW)
    t_load = mine_system.elhd.get_loading_time(vehiculo_prueba)
    t_unload = mine_system.elhd.get_discharging_time(vehiculo_prueba)

    t_acc_out = speed_m_s / acceleration
    t_const_out = (dist_ida - 0.5 * acceleration * t_acc_out**2) / speed_m_s
    t_acc_ret = t_acc_out
    t_const_ret = (dist_vuelta - 0.5 * acceleration * t_acc_ret**2) / speed_m_s

    # -------------------------------
    # Tiempos parciales
    tiempo_ida_h = (t_acc_out + t_const_out) / 3600
    tiempo_vuelta_h = (t_acc_ret + t_const_ret) / 3600
    tiempo_carga_descarga_h = (t_load + t_unload) / 3600

    # -------------------------------
    # Energías específicas
    energia_cinetica = 0.5 * (
        (mine_system.elhd.get_weight(vehiculo_prueba) * 1000) * speed_m_s**2 +  # ida (descargado)
        ((mine_system.elhd.get_weight(vehiculo_prueba) + mine_system.elhd.get_load_capacity(vehiculo_prueba)) * 1000) * speed_m_s**2  # vuelta (cargado)
    ) / 3.6e6

    energia_aerodinamica = mine_system.elhd.aerodynamic_loss(
        vehiculo_prueba, t_acc_out, t_const_out, t_acc_ret, t_const_ret) / 3.6e6

    energia_gravitacional = mine_system.elhd.gravitational_work(
        vehiculo_prueba, tilt, t_acc_out, t_const_out, t_acc_ret, t_const_ret) / 3.6e6

    energia_rodadura = mine_system.elhd.rolling_resistance_loss(
        vehiculo_prueba, tilt, t_acc_out, t_const_out, t_acc_ret, t_const_ret) / 3.6e6

    energia_auxiliar = aux_power_kW * (tiempo_ida_h + tiempo_vuelta_h + tiempo_carga_descarga_h)  # kWh
    energia_hidraulica = hydraulic_power_kW * tiempo_carga_descarga_h  # kWh

    # -------------------------------
    # Suma total componentes
    suma_total_componentes = (
        energia_cinetica + energia_aerodinamica + energia_gravitacional + energia_rodadura +
        energia_auxiliar + energia_hidraulica
    )

    resultados.append({
        "Nodo": nodo,
        "Distancia ida [m]": dist_ida,
        "Distancia vuelta [m]": dist_vuelta,
        "Energía cinética [kWh]": energia_cinetica,
        "Energía aerodinámica [kWh]": energia_aerodinamica,
        "Energía gravitacional [kWh]": energia_gravitacional,
        "Energía rodadura [kWh]": energia_rodadura,
        "Energía auxiliar (todo viaje) [kWh]": energia_auxiliar,
        "Energía hidráulica (carga/descarga) [kWh]": energia_hidraulica,
        "Suma total componentes [kWh]": suma_total_componentes,
        "Energía total consumida [kWh]": energia_total,
        "Diferencia [kWh]": abs(suma_total_componentes - energia_total),
        "Tiempo de viaje [h]": t_viaje
    })

import pandas as pd
df_resultados = pd.DataFrame(resultados)
df_resultados


Vehículo seleccionado: Diesel_1
4
100
4
100
4
100
4
100
4
100


,Nodo,Distancia ida [m],Distancia vuelta [m],Energía cinética [kWh],Energía aerodinámica [kWh],Energía gravitacional [kWh],Energía rodadura [kWh],Energía auxiliar (todo viaje) [kWh],Energía hidráulica (carga/descarga) [kWh],Suma total componentes [kWh],Energía total consumida [kWh],Diferencia [kWh],Tiempo de viaje [h]
0,ex_node_0,346.560,40.050,0.522977,0.022804,0.0,1.702413,0.134112,1.111111,3.493416,3.709108,0.215692,0.053281
1,ex_node_1,326.055,62.070,0.522977,0.022901,0.0,1.741255,0.134415,1.111111,3.532659,3.759206,0.226547,0.053357
2,ex_node_10,150.075,238.275,0.522977,0.022915,0.0,2.001498,0.134460,1.111111,3.792961,4.092060,0.299099,0.053368
3,ex_node_100,139.275,246.825,0.522977,0.022771,0.0,2.004514,0.134010,1.111111,3.795383,4.095283,0.299900,0.053256
4,ex_node_101,224.610,161.490,0.522977,0.022771,0.0,1.878944,0.134010,1.111111,3.669813,3.934707,0.264894,0.053256


In [7]:
time_series.get_marginal_cost(mine_system.battery.get_energy_cost(), 1, 2)

name
Profile_Chuquicamata110    86.81
Profile_Chuquicamata110    86.81
Profile_Chuquicamata110    86.81
Profile_Chuquicamata110    86.81
Profile_Chuquicamata110    86.81
Name: 1, dtype: float64